In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
!unzip -qo /content/drive/MyDrive/reuse_metricgan_finetune.zip -d /content/
%cd /content/reuse_metricgan_finetune
!ls

from huggingface_hub import snapshot_download
snapshot_download(repo_id="nvidia/RE-USE", local_dir="./REUSE", local_dir_use_symlinks=False)
print("RE-USE downloaded.")

if not os.path.exists('./vox-profile-release/src'):
    !git clone https://github.com/tiantiaf0627/vox-profile-release.git ./vox-profile-release
snapshot_download(repo_id="tiantiaf/whisper-large-v3-voice-quality", local_dir="./vox-profile-release", local_dir_use_symlinks=False)
print("Voice quality model downloaded.")


In [ ]:
import torch, sys, re, subprocess
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

!pip install -q huggingface_hub safetensors soundfile pyyaml tqdm loralib
!pip install -q transformers==4.40.0

torch_ver = re.match(r'(\d+\.\d+)', torch.__version__).group(1)
py_ver = f"cp{''.join(str(x) for x in sys.version_info[:2])}"
cxx_abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
print(f"Detected: torch{torch_ver}, {py_ver}, cxx11abi{cxx_abi}")

def try_install_wheel(urls, fallback_pip):
    for url in urls:
        print(f"  Trying: {url.split('/')[-1]}")
        result = subprocess.run([sys.executable, "-m", "pip", "install", url], capture_output=True, text=True)
        if result.returncode == 0:
            print("  OK!")
            return True
        print("  Failed")
    print("  Building from source...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + fallback_pip)
    return True

causal_urls = [
    f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1%2Bcu12torch{torch_ver}cxx11abi{cxx_abi}-{py_ver}-{py_ver}-linux_x86_64.whl"
]
mamba_urls = [
    f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1%2Bcu12torch{torch_ver}cxx11abi{cxx_abi}-{py_ver}-{py_ver}-linux_x86_64.whl"
]
print("\nInstalling causal-conv1d:")
try_install_wheel(causal_urls, ["causal-conv1d>=1.4.0", "--no-build-isolation"])
print("\nInstalling mamba-ssm:")
try_install_wheel(mamba_urls, ["mamba-ssm", "--no-build-isolation"])

print("\n" + "="*60)
print("  ALL INSTALLED.")
print("  NOW: Runtime -> Restart runtime")
print("  Then run Cell 3 onward. Do NOT re-run Cells 1-2.")
print("="*60)


In [ ]:
import torch
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}")

import transformers
print(f"transformers: {transformers.__version__}")
assert transformers.__version__.startswith("4.40"), f"Wrong transformers! Got {transformers.__version__}, need 4.40.x"

import mamba_ssm
print(f"mamba-ssm: {mamba_ssm.__version__}")
print("\nEnvironment OK!")


In [ ]:
import os
import numpy as np
import soundfile as sf

%cd /content/reuse_metricgan_finetune

os.makedirs("./test_data/noisy", exist_ok=True)
os.makedirs("./test_data/clean", exist_ok=True)

SR = 16000
DURATION = 4

np.random.seed(42)
for i in range(20):
    t = np.linspace(0, DURATION, SR * DURATION, dtype=np.float32)
    f0 = np.random.uniform(100, 300)
    clean = np.zeros_like(t)
    for h in range(1, 6):
        clean += (0.3 / h) * np.sin(2 * np.pi * f0 * h * t + np.random.uniform(0, 2*np.pi))
    clean = clean / (np.abs(clean).max() + 1e-8) * 0.5

    snr_db = np.random.uniform(0, 10)
    noise = np.random.randn(len(t)).astype(np.float32)
    noise_level = np.sqrt(np.mean(clean**2)) / (10 ** (snr_db / 20))
    noisy = clean + noise_level * noise

    sf.write(f"./test_data/clean/sample_{i:03d}.wav", clean, SR)
    sf.write(f"./test_data/noisy/sample_{i:03d}.wav", noisy, SR)

print(f"Created 20 synthetic pairs in ./test_data/")


In [ ]:
import sys, os, time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import soundfile as sf
from huggingface_hub import hf_hub_download

device = torch.device('cuda')


sys.path.insert(0, os.path.abspath('./REUSE'))
sys.path.insert(0, os.path.abspath('./vox-profile-release'))


from models.stfts import mag_phase_stft, mag_phase_istft
from models.generator_SEMamba_time_d4 import SEMamba
from utils.util import load_config, pad_or_trim_to_match

config_path = hf_hub_download(repo_id='nvidia/RE-USE', filename='config.json')
cfg = load_config(config_path)
SE_model = SEMamba.from_pretrained('nvidia/RE-USE', cfg=cfg).to(device)
SE_model.train()
print(f"RE-USE loaded ({sum(p.numel() for p in SE_model.parameters())/1e6:.1f}M params)")


from discriminator import MetricDiscriminator
D = MetricDiscriminator(n_fft=512, hop_length=128).to(device)
print(f"Discriminator loaded ({sum(p.numel() for p in D.parameters())/1e3:.1f}K params)")


from vq_metric import VOICE_QUALITY_LABELS, NUM_LABELS
from src.model.voice_quality.whisper_voice_quality import WhisperWrapper

vq_model = WhisperWrapper.from_pretrained("tiantiaf/whisper-large-v3-voice-quality").float().to(device)
vq_model.eval()
for p in vq_model.parameters():
    p.requires_grad_(False)
print(f"Voice quality model loaded (frozen). {NUM_LABELS} labels.")


RELU = nn.ReLU()
n_fft = cfg['stft_cfg']['n_fft']
hop_size = cfg['stft_cfg']['hop_size']
win_size = cfg['stft_cfg']['win_size']
compress_factor = cfg['model_cfg']['compress_factor']
sampling_rate = cfg['stft_cfg']['sampling_rate']

def make_even(v):
    v = int(round(v))
    return v if v % 2 == 0 else v + 1

def enhance(wav_batch):
    sr = 16000
    n_fft_s = make_even(n_fft * sr // sampling_rate)
    hop_s = make_even(hop_size * sr // sampling_rate)
    win_s = make_even(win_size * sr // sampling_rate)
    noisy_mag, noisy_pha, _ = mag_phase_stft(wav_batch, n_fft=n_fft_s, hop_size=hop_s,
                                              win_size=win_s, compress_factor=compress_factor,
                                              center=True, addeps=False)
    amp_g, pha_g, _ = SE_model(noisy_mag, noisy_pha)
    mag = torch.expm1(RELU(amp_g))
    zero_portion = torch.sum(mag == 0, 1) / mag.shape[1]
    amp_g[:, :, (zero_portion > 0.5)[0]] = 0
    audio_g = mag_phase_istft(amp_g, pha_g, n_fft_s, hop_s, win_s, compress_factor)
    audio_g = pad_or_trim_to_match(wav_batch.detach(), audio_g, pad_value=1e-8)
    return audio_g

@torch.no_grad()
def compute_vq_distance(enhanced_wav, clean_wav):
    def get_probs(wav):
        if wav.shape[0] < 16000:
            return None
        data = wav[:15*16000].unsqueeze(0).float().to(device)
        logits = vq_model(data, return_feature=False)
        return torch.sigmoid(logits).cpu().numpy()[0]
    p1 = get_probs(enhanced_wav.cpu())
    p2 = get_probs(clean_wav.cpu())
    if p1 is None or p2 is None:
        return 0.5
    return float(np.abs(p1 - p2).sum() / NUM_LABELS)


class STFTLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x, y):
        window = torch.hann_window(512, device=x.device)
        x_spec = torch.stft(x, 512, 128, 512, window=window, return_complex=True).abs()
        y_spec = torch.stft(y, 512, 128, 512, window=window, return_complex=True).abs()
        sc = torch.norm(y_spec - x_spec, p="fro") / (torch.norm(y_spec, p="fro") + 1e-8)
        mag = nn.functional.l1_loss(torch.log(x_spec + 1e-8), torch.log(y_spec + 1e-8))
        return sc + mag

stft_loss_fn = STFTLoss()
l1_loss_fn = nn.L1Loss()


noisy_files = sorted(os.listdir("./test_data/noisy"))
clean_files = sorted(os.listdir("./test_data/clean"))

def load_batch(indices):
    noisy_list, clean_list = [], []
    for i in indices:
        n, _ = sf.read(f"./test_data/noisy/{noisy_files[i]}", dtype='float32')
        c, _ = sf.read(f"./test_data/clean/{clean_files[i]}", dtype='float32')
        noisy_list.append(torch.from_numpy(n))
        clean_list.append(torch.from_numpy(c))
    return torch.stack(noisy_list).to(device), torch.stack(clean_list).to(device)


opt_g = optim.AdamW(SE_model.parameters(), lr=1e-5, weight_decay=0.01)
opt_d = optim.AdamW(D.parameters(), lr=1e-4, weight_decay=0.01)


print("\nComputing baseline VQ distance (before fine-tuning)...")
baseline_dists = []
with torch.no_grad():
    for i in range(min(10, len(noisy_files))):
        noisy_batch, clean_batch = load_batch([i])
        enhanced = enhance(noisy_batch)
        dist = compute_vq_distance(enhanced[0], clean_batch[0])
        baseline_dists.append(dist)
baseline_mean = np.mean(baseline_dists)
print(f"  Baseline mean VQ distance: {baseline_mean:.4f}")


print(f"\nTraining for 50 steps (batch_size=2)...")
history = {'step': [], 'loss_g': [], 'loss_spec': [], 'loss_gan': [], 'loss_d': [], 'vq_dist': []}

N_STEPS = 50
BATCH_SIZE = 2
N_FILES = len(noisy_files)

for step in range(N_STEPS):
    SE_model.train()
    indices = np.random.choice(N_FILES, BATCH_SIZE, replace=False).tolist()
    noisy_batch, clean_batch = load_batch(indices)

    enhanced = enhance(noisy_batch)

    loss_l1 = l1_loss_fn(enhanced, clean_batch)
    loss_stft = stft_loss_fn(enhanced, clean_batch)
    loss_spec = loss_l1 + loss_stft

    if step % 5 == 0:
        vq_dist = compute_vq_distance(enhanced[0].detach(), clean_batch[0])
        vq_dists_batch = torch.tensor([
            compute_vq_distance(enhanced[j].detach(), clean_batch[j])
            for j in range(BATCH_SIZE)
        ], device=device)

        opt_d.zero_grad()
        pred = D(enhanced.detach(), clean_batch).squeeze(-1)
        loss_d = nn.functional.mse_loss(pred, vq_dists_batch)
        loss_d.backward()
        opt_d.step()
    else:
        vq_dist = history['vq_dist'][-1] if history['vq_dist'] else baseline_mean
        loss_d = torch.tensor(0.0)

    opt_g.zero_grad()
    pred_g = D(enhanced, clean_batch).squeeze(-1)
    loss_gan = nn.functional.mse_loss(pred_g, torch.zeros_like(pred_g))
    loss_g = loss_spec + 0.5 * loss_gan
    loss_g.backward()
    torch.nn.utils.clip_grad_norm_(SE_model.parameters(), 5.0)
    opt_g.step()

    history['step'].append(step)
    history['loss_g'].append(loss_g.item())
    history['loss_spec'].append(loss_spec.item())
    history['loss_gan'].append(loss_gan.item())
    history['loss_d'].append(loss_d.item())
    history['vq_dist'].append(vq_dist)

    if step % 10 == 0:
        print(f"  Step {step:3d}: L_g={loss_g.item():.4f} L_spec={loss_spec.item():.4f} "
              f"L_gan={loss_gan.item():.4f} VQ_dist={vq_dist:.4f}")

print("\nTraining done!")


In [ ]:
print("Computing final VQ distance (after 50 steps)...")
final_dists = []
SE_model.eval()
with torch.no_grad():
    for i in range(min(10, len(noisy_files))):
        noisy_batch, clean_batch = load_batch([i])
        enhanced = enhance(noisy_batch)
        dist = compute_vq_distance(enhanced[0], clean_batch[0])
        final_dists.append(dist)
final_mean = np.mean(final_dists)

print(f"\n{'='*60}")
print(f"  RESULTS: 50-step mini fine-tuning test")
print(f"{'='*60}")
print(f"  Baseline VQ distance:   {baseline_mean:.4f}")
print(f"  After 50 steps:         {final_mean:.4f}")
print(f"  Change:                 {final_mean - baseline_mean:+.4f} ({'improved' if final_mean < baseline_mean else 'no improvement yet'})")
print(f"  Spectral loss (start):  {history['loss_spec'][0]:.4f}")
print(f"  Spectral loss (end):    {history['loss_spec'][-1]:.4f}")
print(f"{'='*60}")

if final_mean < baseline_mean:
    print("\n  LEARNING SIGNAL DETECTED -- VQ distance improved!")
    print("  The pipeline works. Ready for full training on school GPU.")
else:
    print("\n  VQ distance did not improve in 50 steps.")
    print("  This is expected with synthetic data and very few steps.")
    print("  Check that spectral loss decreased (denoising is learning).")
    print("  With real data + more steps, VQ should improve too.")
